# M3.3 · Tensor autograd motoru (MiniTorch tarzı)

micrograd'ın skalerlerini tensörlere büyütüyorsun. Broadcasting'li bir işlemin gradyanını elle "geri toplamayı" (unbroadcast) öğreniyor, kuralları saf Python'da yazıyor, NumPy üzerinde gerçek bir `Tensor` sınıfıyla autograd motoru kurup onunla softmax regression eğitiyor, PyTorch'ta autograd kullanmadan aynı gradyanları elle türetiyor ve en sonda forward-mode ile reverse-mode Jacobian'ın hangisinin ne zaman kazandığını ölçüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-3-tensor-autograd.html](https://m-gonul.github.io/pytorch-lab/m3-3-tensor-autograd.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_3_tensor_autograd.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def bs():\n        f = need(ns, \"broadcast_shape\")\n        for a, b, want in [((2, 3), (3,), (2, 3)), ((4, 1), (1, 5), (4, 5)), ((5, 1, 3), (2, 1), (5, 2, 3)), ((), (3,), (3,)), ((7,), (1,), (7,))]:\n            ensure(tuple(f(a, b)) == want, f\"{a} + {b} → {f(a, b)}, {want} bekleniyordu\")\n        try:\n            f((2, 3), (2,))\n        except ValueError:\n            return\n        raise AssertionError(\"(2, 3) ve (2,) için ValueError bekleniyordu\")\n    R.test(\"broadcast_shape: kurallar ve uyumsuzluk\", bs)\n\n    def ub():\n        f = need(ns, \"unbroadcast2d\")\n        G = [[1, 2, 3], [4, 5, 6]]\n        close(f(G, (3,)), [5, 7, 9]); close(f(G, (1, 3)), [[5, 7, 9]]); close(f(G, (2, 1)), [[6], [15]])\n        close(f(G, ()), 21); close(f(G, (2, 3)), G)\n    R.test(\"unbroadcast2d: (m,), (1, m), (n, 1), (), (n, m)\", ub)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef numgrad(f, arrs, h=1e-6):\n    out = []\n    for a in arrs:\n        g = np.zeros_like(a)\n        it = np.nditer(a, flags=[\"multi_index\"])\n        for _ in it:\n            i = it.multi_index\n            old = a[i]; a[i] = old + h; fp = f(); a[i] = old - h; fm = f(); a[i] = old\n            g[i] = (fp - fm) / (2 * h)\n        out.append(g)\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    T = need(ns, \"Tensor\", \"sınıf\") if \"Tensor\" in ns else None\n    if T is None:\n        R.test(\"`Tensor` tanımlı\", lambda: need(ns, \"Tensor\", \"sınıf\"))\n        return\n    rng = np.random.default_rng(0)\n\n    def ub():\n        f = need(ns, \"unbroadcast\")\n        G = rng.normal(size=(4, 2, 3))\n        close(f(G, (3,)), G.sum((0, 1))); close(f(G, (2, 1)), G.sum(0).sum(1, keepdims=True)); close(f(G, (4, 1, 3)), G.sum(1, keepdims=True)); close(f(G, (4, 2, 3)), G)\n    R.test(\"unbroadcast genel kural\", ub)\n\n    def expr():\n        A, B, c = rng.normal(size=(4, 3)), rng.normal(size=(3, 5)), rng.normal(size=(5,))\n        def f():\n            a, b, cc = T(A), T(B), T(c)\n            z = ((a @ b + cc) * 0.5 - cc ** 2).relu() + (a @ b) / 3.0\n            return z.sum(), (a, b, cc)\n        L, ts = f()\n        L.backward()\n        num = numgrad(lambda: float(f()[0].data), [A, B, c])\n        for t, gnum, name in zip(ts, num, \"ABc\"):\n            close(t.grad, gnum, rtol=1e-5, atol=1e-6, what=f\"d{name}\")\n    R.test(\"matmul + broadcast + relu + pow + div: sayısal gradyanla aynı\", expr)\n\n    def lse():\n        Z0 = rng.normal(size=(6, 4))\n        def f():\n            z = T(Z0)\n            return ((z.exp().sum(axis=1, keepdims=True)).log() - z.sum(axis=1, keepdims=True) * 0.25).mean(), z\n        L, z = f(); L.backward()\n        close(z.grad, numgrad(lambda: float(f()[0].data), [Z0])[0], rtol=1e-5, atol=1e-7, what=\"dZ\")\n    R.test(\"exp / log / sum(axis, keepdims) / mean\", lse)\n\n    def acc():\n        x = T(np.array([1.0, -2.0, 3.0]))\n        (x * x).sum().backward()\n        close(x.grad, [2, -4, 6], what=\"x·x\")\n        y = T(np.array(2.0)); (y + y * y + 1 - y).backward()\n        close(y.grad, 4.0, what=\"y + y² + 1 − y\")\n    R.test(\"paylaşılan düğümde gradyan birikiyor\", acc)\n\n    def train():\n        Xn = np.concatenate([rng.normal([0, 3], 1, (40, 2)), rng.normal([-3, -2], 1, (40, 2)), rng.normal([3, -2], 1, (40, 2))])\n        yn = np.repeat([0, 1, 2], 40)\n        Y = np.eye(3)[yn]\n        W, b = T(np.zeros((2, 3))), T(np.zeros(3))\n        for _ in range(150):\n            W.grad[...] = 0; b.grad[...] = 0\n            Z = T(Xn) @ W + b\n            L = ((Z.exp().sum(axis=1, keepdims=True)).log() - (Z * Y).sum(axis=1, keepdims=True)).mean()\n            L.backward()\n            W.data -= 0.5 * W.grad; b.data -= 0.5 * b.grad\n        a = ((Xn @ W.data + b.data).argmax(1) == yn).mean()\n        ensure(a >= 0.9, f\"doğruluk {a:.2f}\")\n        return f\"senin motorunla softmax regression: doğruluk {a:.3f}, son kayıp {float(L.data):.3f}\"\n    R.test(\"motorla softmax regression eğitimi\", train)\n",
 "d": "import ast\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def noauto():\n        fn = next((n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef) and n.name == \"manual_grads\"), None)\n        ensure(fn is not None, \"`manual_grads` yok\")\n        names = {n.attr for n in ast.walk(fn) if isinstance(n, ast.Attribute)} | {n.id for n in ast.walk(fn) if isinstance(n, ast.Name)}\n        for bad in (\"backward\", \"autograd\", \"func\", \"requires_grad\", \"requires_grad_\", \"sum_to_size\", \"grad\"):\n            ensure(bad not in names, f\"manual_grads içinde `{bad}` kullanılmamalı\")\n    R.test(\"manual_grads autograd kullanmıyor\", noauto)\n    g = torch.Generator().manual_seed(0)\n\n    def ub():\n        f = need(ns, \"unbroadcast_torch\")\n        G = torch.randn(4, 2, 3, generator=g)\n        close(f(G, (3,)), G.sum((0, 1))); close(f(G, (2, 1)), G.sum(0).sum(1, keepdim=True)); close(f(G, torch.Size([4, 1, 3])), G.sum(1, keepdim=True))\n    R.test(\"unbroadcast_torch\", ub)\n\n    def mg():\n        a = torch.randn(7, 5, generator=g, dtype=torch.float64); b = torch.randn(5, generator=g, dtype=torch.float64); W = torch.randn(5, 4, generator=g, dtype=torch.float64)\n        da, db, dW = need(ns, \"manual_grads\")(a, b, W)\n        A, B, Wt = (t.clone().requires_grad_(True) for t in (a, b, W))\n        torch.relu((A * B + A) @ Wt).sum().backward()\n        close(da, A.grad, what=\"da\"); close(db, B.grad, what=\"db (unbroadcast!)\"); close(dW, Wt.grad, what=\"dW\")\n        ensure(tuple(db.shape) == (5,), f\"db shape {tuple(db.shape)}, (5,) bekleniyordu\")\n    R.test(\"manual_grads = autograd\", mg)\n",
 "e": "import torch\nfrom torch.func import jacrev, jacfwd\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: (max_loops(src, 0, func=\"jacobian_rev\"), max_loops(src, 0, func=\"jacobian_fwd\")))\n    g = torch.Generator().manual_seed(0)\n    A = torch.randn(6, 5, generator=g, dtype=torch.float64)\n    f = lambda x: torch.tanh(A @ x) * x.sum()\n    x = torch.randn(5, generator=g, dtype=torch.float64)\n    R.test(\"jacobian_rev = jacrev\", lambda: close(need(ns, \"jacobian_rev\")(f, x), jacrev(f)(x)))\n    R.test(\"jacobian_fwd = jacfwd\", lambda: close(need(ns, \"jacobian_fwd\")(f, x), jacfwd(f)(x)))\n    if \"jacobian_rev\" not in ns or \"jacobian_fwd\" not in ns:\n        return\n    rows = []\n    for n, m in ((8, 1024), (1024, 8)):\n        W1 = torch.randn(256, n, generator=g) / n ** 0.5\n        W2 = torch.randn(m, 256, generator=g) / 16\n        h = lambda x: W2 @ torch.tanh(W1 @ x)\n        xx = torch.randn(n, generator=g)\n        tr = bench(lambda: ns[\"jacobian_rev\"](h, xx), repeat=3)\n        tf = bench(lambda: ns[\"jacobian_fwd\"](h, xx), repeat=3)\n        rows.append([f\"n = {n}, m = {m}\", fmt_time(tr), fmt_time(tf), \"forward\" if tf < tr else \"reverse\"])\n    R.table(\"Tam Jacobian: reverse (m VJP) vs forward (n JVP)\", [\"boyutlar\", \"reverse\", \"forward\", \"kazanan\"], rows,\n            note=\"Kayıp fonksiyonu m = 1: tek VJP. Backprop'un reverse-mode olmasının nedeni bu.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m3_3_tensor_autograd as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

micrograd her sayı için bir düğüm kuruyordu. Gerçek motorlar bir düğümü bir **tensör** yapar ve her işlem için tensör düzeyinde bir backward kuralı tanımlar. Gelen gradyan $G=\partial L/\partial Y$ ile:

| ileri | geri |
|---|---|
| $Y=A+B$ | $dA=\text{unb}(G)$, $dB=\text{unb}(G)$ |
| $Y=A\odot B$ | $dA=\text{unb}(G\odot B)$, $dB=\text{unb}(G\odot A)$ |
| $Y=AB$ | $dA=GB^\top$, $dB=A^\top G$ |
| $y=\sum_{\text{eksen}}A$ | $dA=G$ (toplanan eksende yayılmış) |
| $Y=f(A)$ eleman eleman | $dA=G\odot f'(A)$ |

**unb** = unbroadcast: ileri geçişte broadcasting bir girdiyi yaydıysa, gelen gradyanı o eksenler üzerinden toplayarak girdinin shape'ine geri indir. Kural: (1) baştaki fazla eksenler üzerinden topla, (2) girdide boyutu 1 olan eksenler üzerinden `keepdims=True` ile topla.

Reverse-mode (backprop), bir **vektör–Jacobian çarpımı** (VJP) hesaplar: $v^\top J$. Forward-mode ise **Jacobian–vektör çarpımı** (JVP): $Jv$. Tam Jacobian ya $m$ VJP ya $n$ JVP ister.

## 1 · Elle

Broadcasting'i "kopyalama" olarak düşün: $b$ iki kez kopyalanıp $a$'nın iki satırıyla çarpıldı. Her kopya kayba katkı yaptı; $b$'nin gradyanı kopyaların gradyanlarının toplamı. $c$ $(2,1)$ ise üç sütuna kopyalandı; gradyanı satır başına 3.

Matmul'de shape kuralı her şeyi söyler: $dA$, $A$ ile aynı shape'te olmalı ve $G=\mathbf 1_{2\times2}$ ile $GB^\top$ bunu verir.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$a=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$ $(2\times3)$, $b=[1,1,1]$ $(3,)$, $L=\sum(a\odot b)$**

- `db1`: $\partial L/\partial b_1$
- `db3`: $\partial L/\partial b_3$
- `da`: $\partial L/\partial a_{23}$

**$L=\sum(a+c)$, $c$ shape'i $(2,1)$**

- `dc`: $\partial L/\partial c_1$

**$L=\sum(AB)$, $A$ $(2\times3)$, $B=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix}$**

- `dA11`: $\partial L/\partial A_{11}$
- `dA13`: $\partial L/\partial A_{13}$
- `rows`: $dA$'nın satır sayısı

In [ ]:
cevaplar = dict(
    db1=None,
    db3=None,
    da=None,
    dc=None,
    dA11=None,
    dA13=None,
    rows=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"db1":[5,0],"db3":[9,0],"da":[1,0],"dc":[3,0],"dA11":[3,0],"dA13":[11,0],"rows":[2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; shape'ler tuple, 2-boyutlu gradyanlar liste listesi.

- `broadcast_shape(s1, s2)` — NumPy kuralı: sağdan hizala, her eksende eşit ya da biri 1 olmalı, eksik eksen 1 sayılır. Uyumsuzsa `ValueError`.
- `unbroadcast2d(G, shape)` — `G` $(n\times m)$ liste; `shape` şunlardan biri: `(n, m)`, `(m,)`, `(1, m)`, `(n, 1)`, `()`. Dönüş: `(m,)` → liste, `(1, m)` → `[liste]`, `(n, 1)` → `[[s], [s], …]`, `()` → sayı.

In [ ]:
%%writefile stage_b.py
def broadcast_shape(s1, s2):
    ...


def unbroadcast2d(G, shape):
    ...


print(broadcast_shape((2, 3), (3,)), broadcast_shape((4, 1), (1, 5)))   # (2, 3) (4, 5)
G = [[1, 2, 3], [4, 5, 6]]
print(unbroadcast2d(G, (3,)), unbroadcast2d(G, (2, 1)), unbroadcast2d(G, ()))   # [5, 7, 9] [[6], [15]] 21

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy üzerinde bir autograd motoru. `Tensor(data)`: `self.data` (float64 `ndarray`), `self.grad` (aynı shape, sıfırla başlar), `self._prev`, `self._backward`.

Yazılacaklar (hepsi tensör düzeyinde, **eleman döngüsü yok**):

- `unbroadcast(grad, shape)` — b'deki kuralın genel hâli (`grad.sum(axis=…, keepdims=…)`).
- `__add__`, `__mul__`, `__matmul__`, `__neg__`, `__sub__`, `__pow__(k)` (sayı), `__truediv__`, `__radd__`, `__rmul__`.
- `sum(axis=None, keepdims=False)`, `mean()`, `relu()`, `exp()`, `log()`.
- `backward()` — topolojik sıra, `self.grad = np.ones_like(self.data)`, ters sırada `_backward`.

Sayı ile işlemler (`t * 2`, `1 - t`) sayıyı `Tensor`'a sarsın. Kontrol: rastgele ifadeler sayısal gradyanla, `x * x` birikimi, ve motorunla eğitilen softmax regression.

In [ ]:
%%writefile stage_c.py
import numpy as np


def unbroadcast(grad, shape):
    ...


class Tensor:
    def __init__(self, data, _prev=(), _op=""):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._prev = set(_prev)
        self._op = _op
        self._backward = lambda: None

    @staticmethod
    def wrap(x):
        return x if isinstance(x, Tensor) else Tensor(x)

    def __add__(self, other):
        other = Tensor.wrap(other)
        out = Tensor(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += unbroadcast(out.grad, self.data.shape)
            other.grad += unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    # __mul__, __matmul__, __neg__, __sub__, __pow__, __truediv__, __radd__, __rmul__, __rsub__
    # sum, mean, relu, exp, log, backward
    ...


x = Tensor([[1.0, 2, 3], [4, 5, 6]])
b = Tensor([1.0, 1, 1])
L = (x * b).sum()
L.backward()
print(b.grad)        # [5. 7. 9.]

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Şimdi PyTorch'ta, ama **autograd'sız**: gradyanları tensör işlemleriyle elle yaz ve autograd'ın bulduğuyla karşılaştır.

$$
Z=(a\odot b+a)\,W,\qquad L=\sum\text{ReLU}(Z),\qquad a:(n,d),\ b:(d,),\ W:(d,k).
$$

- `unbroadcast_torch(G, shape)` — genel kural, `sum_to_size` kullanmadan.
- `manual_grads(a, b, W)` → `(da, db, dW)`; `backward`, `torch.autograd` ya da `torch.func` **yasak**.

In [ ]:
%%writefile stage_d.py
import torch


def unbroadcast_torch(G, shape):
    ...


def manual_grads(a, b, W):
    """L = ReLU((a * b + a) @ W).sum() için (da, db, dW) — autograd YOK."""
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

$f:\mathbb R^n\to\mathbb R^m$ için tam Jacobian:

- **reverse:** $m$ tane VJP, her biri bir satır: `vjp_fn(e_i)`; birim vektörler üzerinden `vmap`.
- **forward:** $n$ tane JVP, her biri bir sütun: `jvp(f, (x,), (e_j,))`; birim vektörler üzerinden `vmap`.

```python
from torch.func import vjp, jvp
y, vjp_fn = vjp(f, x)
```

- `jacobian_rev(f, x)` → `(m, n)`; `vjp` + `vmap`, **döngü yok**.
- `jacobian_fwd(f, x)` → `(m, n)`; `jvp` + `vmap`, **döngü yok**.

Kontrol, `torch.func.jacrev`/`jacfwd` ile karşılaştırır ve iki uç durumu ($n=8,m=1024$ ve $n=1024,m=8$) yarıştırır.

In [ ]:
%%writefile stage_e.py
import torch
from torch.func import vjp, jvp


def jacobian_rev(f, x):
    """→ (m, n), vjp + vmap. Döngü yok."""
    ...


def jacobian_fwd(f, x):
    """→ (m, n), jvp + vmap. Döngü yok."""
    ...


A = torch.randn(3, 4)
f = lambda x: torch.tanh(A @ x)
x = torch.randn(4)
print(jacobian_rev(f, x).shape, jacobian_fwd(f, x).shape)   # (3, 4) (3, 4)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Paszke ve ark. (2019): PyTorch

[Paszke ve ark. — PyTorch: An Imperative Style, High-Performance Deep Learning Library (NeurIPS 2019)](https://arxiv.org/abs/1912.01703)

Makalenin Bölüm 5'teki performans iddialarını kendi makinende sına: (1) Python yükünü — aynı küçük MLP'yi farklı batch boyutlarında çalıştırıp çekirdek süresi ile Python/dispatch süresini torch.profiler ile ayır (Şekil 1'deki gibi), (2) önbellekli ayırıcının (caching allocator) etkisini — PYTORCH_NO_CUDA_MEMORY_CACHING ya da CPU'da sık tensör ayırma ile ölç, (3) referans sayımlı serbest bırakmanın bellek tepe noktasına etkisi. Küçük batch'te yükün baskın olduğunu ve CUDA graphs / torch.compile'ın bunu nasıl azalttığını (M6.5) göster.


**Adımlar**

1. Profili Chrome trace olarak kaydet ve yorumla.

**Kabul ölçütleri**

- Batch boyutuna göre yük/çekirdek oranı grafiği.
- Ayırıcı ve serbest bırakma ölçümleri.

## Kaynaklar

- [MiniTorch — Module 1 (Autodiff) ve Module 2 (Tensors)](https://minitorch.github.io/) — lisans dosyası yok
- [Karpathy — micrograd](https://github.com/karpathy/micrograd) — MIT
- [CS231n — Derivatives, Backpropagation, and Vectorization (notlar)](https://cs231n.stanford.edu/handouts/derivatives.pdf) — ©
- [PyTorch — torch.func: jvp, vjp, jacfwd, jacrev](https://docs.pytorch.org/docs/stable/func.api.html) — BSD-3